# IAA Hackathon 2026 — Notebook 04: Decision Log & Audit Trail

Comprehensive record of all AI and human decisions, assumptions, modelling approaches, and results across the four-notebook pipeline (v4). This notebook serves as the audit trail for the hackathon submission.

## Pipeline Overview

| Notebook | Title | Purpose | Status |
| --- | --- | --- | --- |
| 01 | Data Prep & Claims Development | Data import, cleaning, enrichment joins, claims-to-ultimate via BF, IBNR triangles, seasonality/trend analysis | Complete |
| 02 | XGBoost Tweedie | Feature selection from Features_v3.xlsx, label encoding, XGBoost reg:tweedie (p=1.5) with base_margin offset, validation, submission, trend uplift | Complete |
| 03 | GLM & Blend | GLM Tweedie with target encoding, XGBoost importance-based feature selection, Year forced into GLM, optimised per-line blend weights, submissions | Complete |
| 04 | Decision Log | Audit trail of all decisions, assumptions, results, future improvements | This notebook |

**Key design principles (human):**
* Use pandas (not Spark) for all operations
* Checkpoint all intermediate results to Excel files
* Keep run times manageable (minutes, not hours)
* Reproducible, modular pipeline with clear documentation
* Defer k-fold and hyperparameter tuning unless needed after initial submissions

## v4 Key Updates (Oct 24)

### 1. BF Development Factor Correction (Critical Fix)

**Problem identified by human teammate**: Motor predictions were much lower than teammate's. Root cause was insufficient BF development factors for 2024, particularly bodily injury claims.

**Analysis**: Bodily injury claims for 2024 were only 54.9% developed at lag0 (1,178 claims reported), but 2021 at the same maturity point also had 1,178 claims that eventually developed to 2,146. The old factor of 1.30 only brought 2024 to 1,531 claims — 72% of the true ultimate.

**Fix**: Replaced assumed development factors with chain-ladder derived per-peril factors from the 2021 mature year:

| Peril tail | Old 2024 factor | New 2024 factor | Chain-ladder basis |
| --- | --- | --- | --- |
| Bodily injury | 1.30 | **1.82** | 2,146/1,178 = 1.82 (54.9% at lag0) |
| Third party prop | 1.30 | **1.08** | 8,193/7,573 = 1.08 (92.5% at lag0) |
| Short motor (OD, theft, glass) | 1.10 | **1.08** | ~95% at lag0 |
| Subsidence | 1.15 | **1.47** | 283/193 = 1.47 (68.2% at lag0) |
| Buildings other | 1.15 | **1.08** | 93-98% at lag0 |
| Contents | 1.15 | **1.04** | 92-98% at lag0 |

Also fixed 2023 over-development: old long-tail factor 1.15 → 1.06 for BI (94.5% at lag0+1), 1.01 for TPP.

**Impact on claims rates**:

| Line | 2021 | 2022 | 2023 | 2024 | Old 2024 | Old CAGR | New CAGR |
| --- | --- | --- | --- | --- | --- | --- | --- |
| Motor | 563 | 590 | 605 | 568 | 552 | 8.0% (false) | 0.3% |
| Buildings | 384 | 420 | 449 | 506 | 517 | 10.4% | 9.6% |
| Contents | 106 | 108 | 118 | 121 | 134 | 8.4% | 4.9% |

### 2. Features_v3.xlsx (skiprows fix)
- v2 had inconsistent skiprows across sheets (motor=0, buildings=5, contents=2) causing Year to be missed for buildings/contents
- v3 has uniform layout, skiprows=0 for all sheets, Year marked Include=1 for all three lines
- Motor: 65 features, Buildings: 54, Contents: 54 (all with Year included)

### 3. Updated Trend Factors
Based on corrected BF claims rates:

| Line | Old trend factor | New trend factor | Basis |
| --- | --- | --- | --- |
| Motor | 1.08 | **1.04** | CAGR 2021-2023 ~3.7% (severity-driven, frequency declining) |
| Buildings | 1.104 | **1.096** | CAGR 2021-2024 = 9.6% (very consistent) |
| Contents | 1.084 | **1.049** | CAGR 2021-2024 = 4.9% |

### 4. Year Forced into GLM
- GLM uses top 25 XGBoost importance features; Year was not in top 25 for motor
- Year forcibly added to GLM features for all lines to capture trend/inflation
- GLM can extrapolate Year linearly to 2025 (unlike XGBoost trees)
- GLM trend uplift: motor ×1.04 (Year coefficient dampened by 2024 dip), buildings/contents ×1.0 (Year in GLM captures full trend)

### 5. Blend Weight Optimisation
Searched XGBoost weight from 0.20 to 0.80 in 0.05 steps per line:

| Line | Optimal XGB weight | D² |
| --- | --- | --- |
| Motor | 0.60 | 0.0671 |
| Buildings | 0.40 | 0.0995 |
| Contents | 0.35 | 0.0868 |
| **Average** | — | **0.0844** |

### 6. Final Results

| Line | XGB D² | GLM D² | Blend 50/50 | Per-line optimal |
| --- | --- | --- | --- | --- |
| Motor | 0.0599 | 0.0577 | 0.0667 | 0.0671 |
| Buildings | 0.0588 | 0.0905 | 0.0990 | 0.0995 |
| Contents | 0.0470 | 0.0826 | 0.0857 | 0.0868 |
| **Average** | 0.0552 | 0.0769 | 0.0838 | **0.0844** |

## Notebook 01: Data Prep & Claims Development

### Data Cleaning Decisions

| # | Decision | Source | Rationale |
| --- | --- | --- | --- |
| 1 | Recalculated exposure = (period_end - period_start + 1) / 365 | AI | Original exposure field was unreliable; recalculation ensures consistency across all policy files |
| 2 | Created Year field from period_start | AI | Needed for weather join (address_id + Year) and time-based train/validate split |
| 3 | Normalised construction_type: .str.upper(), W and T combined as TIMBER | AI (confirmed by human) | Wood and Timber are the same construction type; uppercasing ensures consistency (BRICK, TIMBER, NON_STANDARD, STONE) |
| 4 | Stripped whitespace from all ID columns (policy_id, customer_id, address_id, claim_id, row_id) | AI | IDs with trailing/leading spaces cause join failures |
| 5 | Zero-padded motor customer_id to 8 chars | AI | ~10% of motor customer_ids had varying lengths, causing customer join failures |
| 6 | Dropped compulsory_excess (constant) | AI | No predictive value — same value for all rows |
| 7 | Fixed sum_insured x1000 errors in buildings/contents (values > 100x median divided by 1000) | AI | Data entry errors where sum_insured was entered in cents instead of dollars |
| 8 | Flagged but did NOT modify claims with incurred > 100x median | AI | These are genuine large claims (BI, fire, flood), not x1000 errors — a true x1000 error would be ~$2M+ |

### Enrichment Join Decisions

* **Customers**: Left join on customer_id (dropped PII: surname, date_of_birth)
* **Addresses**: Left join on address_id
* **Weather**: Left join on address_id + Year (aggregated to address-year level with mean)
* All joins confirmed 100% match rate, no row blow-up

### Claims Processing Decisions

| # | Decision | Source | Rationale |
| --- | --- | --- | --- |
| 1 | Created incurred_net_of_recoveries = incurred_to_date - recoveries | AI | Net claims cost is the correct target for pricing |
| 2 | Excluded declined/repudiated claims from target | AI | These are not valid claims and should not be predicted |
| 3 | Applied Bornhuetter-Ferguson (BF) development factors by peril tail category and accident year | AI | 2024 claims are immature (only ~1 year of development); BF factors account for IBNR |
| 4 | Development factor categories: long (BI/third party), short_motor (own damage/glass), moderate (property) | AI | Based on general insurance knowledge of claim reporting patterns |
| 5 | DEV_FACTORS: 2021=1.0 (mature), 2022=1.02-1.05, 2023=1.05-1.15, 2024=1.10-1.30 | AI | More immature years get higher uplift factors |
| 6 | Trend uplift = False (rely on BF development only) | AI | Claim count triangle shows 2024 decline is reporting lag, not genuine trend decline |
| 7 | Claims matched to policy rows by policy_id + year(loss_date) + loss_date within [period_start, period_end] | AI | Ensures claims are attributed to the correct policy period |
| 8 | Unmatched claims flagged but not dropped | AI | Transparency — unmatched claims may indicate data quality issues |

### Claims Match Rates
* Motor: ~10% of rows with claims
* Buildings: ~3-4% of rows with claims
* Contents: ~3-4% of rows with claims

### Excel Checkpoint
* File: `Excel files_v4/claims_development_summary.xlsx` (6 sheets: dev factors, claims rates, seasonality, trend, claim count triangle, QA summary)
* Processed data saved to `data/proc/proc_{line}.parquet`

## Notebook 02: XGBoost Tweedie

### Feature Selection Decisions

| # | Decision | Source | Rationale |
| --- | --- | --- | --- |
| 1 | Feature selection from Features_v3.xlsx (Include=1 per line) | Human | User provided explicit feature keep/drop judgements in spreadsheet; v3 fixed skiprows issue from v2 |
| 2 | Stripped whitespace from feature names in spreadsheet | AI | Some column names had extra whitespace causing mismatches with data columns |
| 3 | Only features present in both spreadsheet and data used | AI | Unmatched features logged but not included |
| 4 | compulsory_excess dropped (not in data — was constant) | AI | Dropped in Notebook 1, so not available for modelling |

### Encoding Decisions

* **Categorical variables**: `pd.Categorical` for label consistency (missing values as -1)
* **Booleans**: Converted to 0/1
* **Numerics**: Retained as-is

### XGBoost Modelling Decisions

| # | Decision | Source | Rationale |
| --- | --- | --- | --- |
| 1 | Used DMatrix API with base_margin = log(exposure) for BOTH training and prediction | AI (after initial failure) | Initial sklearn API used base_margin only in predict — predictions were 1000x too low. DMatrix API ensures the exposure offset is correctly applied during training and prediction |
| 2 | reg:tweedie, power=1.5 | Human (from v3) | Tweedie distribution with p=1.5 is standard for insurance pure premium modelling (compound Poisson-Gamma) |
| 3 | Hyperparams: max_depth=6, colsample_bytree=0.5, min_child_weight=10, reg_alpha=0.5, reg_lambda=2.0, n_estimators=300, learning_rate=0.1 | Human (from v3) | Well-regularised parameters that prevent overfitting |
| 4 | No k-fold, no hypertuning | Human | Defer until after initial submissions per user plan |
| 5 | Time-based split: 2021-2023 train, 2024 validate | AI | Mimics the real-world temporal prediction task |
| 6 | sample_weight = exposure | AI | Weight loss by exposure to give more importance to longer-exposure policies |
| 7 | Early stopping (50 rounds) on validation | AI | Prevents overfitting and reduces training time |

### XGBoost Validation D2 Scores

| Line | D2 (Tweedie p=1.5) | Val Pred Mean | Val Actual Mean |
| --- | --- | --- | --- |
| Motor | 0.0674 | 388.53 | 432.80 |
| Buildings | 0.0617 | 271.81 | 420.96 |
| Contents | 0.0402 | 70.79 | 107.44 |
| **Average** | **0.0564** | | |

### Top Features (Motor)
1. ncb_years (No Claims Bonus)
2. driver_age
3. rainfall_mm
4. vehicle_use
5. years_licensed

### Top Features (Buildings)
1. flood_zone
2. occupancy
3. elevation_m
4. missed_payments_12m
5. sum_insured

### Outputs
* Submission: `submissions_v4/xgboost_tweedie_v4.csv` (626,639 rows, mean=250.35)
* Excel: `Excel files_v4/xgboost_v4_results.xlsx` (D2 scores + feature importance per line)
* Validation predictions saved to `data/proc/xgb_val_pred_{line}.parquet` (for blend in Notebook 3)

## Notebook 03: GLM & Blend

### GLM Feature Selection Decisions

| # | Decision | Source | Rationale |
| --- | --- | --- | --- |
| 1 | Use XGBoost feature importance to select GLM features | Human | User instruction: "Use the XGBoost feature significance to determine what factors to include" |
| 2 | Include as many features as possible | Human | User: "Include as many as you can though — I don't want to miss data and then score poorly" |
| 3 | MAX_FEATS = 25 (top features by XGBoost importance) | AI | Balances inclusion with convergence — more than v3 (15) but manageable |
| 4 | Target encoding (Bayesian shrinkage) for ALL categoricals | AI | v3 used one-hot encoding which caused convergence failures with too many factors. Target encoding keeps each categorical as 1 numeric column, avoiding dimensionality explosion |
| 5 | SMOOTHING = 30 for target encoding | AI (from v3) | Bayesian shrinkage parameter — higher = more shrinkage toward overall mean |
| 6 | StandardScaler on all features | AI | Ensures comparable coefficient magnitudes and aids convergence |
| 7 | Only training data (2021-2023) used for target encoding | AI | Prevents data leakage from validation set |

### GLM Modelling Decisions

| # | Decision | Source | Rationale |
| --- | --- | --- | --- |
| 1 | statsmodels GLM Tweedie (power=1.5) with log link | AI | Standard approach for insurance pricing — log link ensures positive predictions |
| 2 | offset = log(exposure) | AI | Exposure offset ensures predictions scale with policy duration |
| 3 | IRLS solver, max_iter=100 | AI | Default iteratively reweighted least squares for GLM |
| 4 | Predictions clipped to [0.01, 50000] | AI | Prevents inf from overflow in exp() and ensures valid submission values |

### Blend Decisions

| # | Decision | Source | Rationale |
| --- | --- | --- | --- |
| 1 | 50/50 blend: (XGB + GLM) / 2 | AI | Equal weighting captures strengths of both models |
| 2 | 70/30 blend: 0.7 * XGB + 0.3 * GLM | AI | XGBoost-weighted blend — tests if tree model should dominate |
| 3 | Validation D2 computed on 2024 holdout for all blends | AI | Allows comparison of blend strategies before submission |

### GLM Validation D2 Scores

| Line | XGB D2 | GLM D2 | Blend 50/50 | Blend 70/30 |
| --- | --- | --- | --- | --- |
| Motor | 0.0674 | 0.0617 | **0.0720** | 0.0729 |
| Buildings | 0.0617 | **0.0922** | **0.1001** | 0.0946 |
| Contents | 0.0402 | **0.0797** | **0.0788** | 0.0716 |
| **Average** | 0.0564 | 0.0779 | **0.0836** | 0.0797 |

### Key Findings
* GLM significantly outperforms XGBoost on buildings (0.0922 vs 0.0617) and contents (0.0797 vs 0.0402)
* XGBoost slightly better on motor (0.0674 vs 0.0617)
* **50/50 blend is the best model** with 0.0836 average D2 — 48% improvement over XGBoost alone, 7% over GLM alone
* The blend captures complementary strengths: XGBoost captures non-linear interactions, GLM provides stable linear relationships

### Outputs
* `submissions_v4/blend_5050_v4.csv` (mean=321.36) — **recommended submission**
* `submissions_v4/blend_7030_v4.csv` (mean=292.96)
* `submissions_v4/glm_tweedie_v4.csv` (mean=392.37)
* `Excel files_v4/glm_blend_v4_results.xlsx` (model comparison + GLM coefficients per line)

## Key Assumptions

### Claims Development (BF Method)
1. Development factors are chain-ladder derived from the 2021 mature year. Per-peril: BI=1.82, TPP=1.08, subsidence=1.47, buildings_other=1.08, contents=1.04 for 2024
2. Six tail categories: bi (very long), tpp (moderate), short_motor, subsidence (long), buildings_other, contents
3. 2021 is treated as fully mature (4 years of development data)
4. Trend uplift applied to 2025 predictions: motor ×1.04, buildings ×1.096, contents ×1.049
5. GLM has Year feature and extrapolates trend; explicit GLM uplift only for motor (×1.04)
6. Declined/repudiated claims excluded from target (not valid claims)

### Modelling
1. Tweedie power=1.5 assumed appropriate for all three lines (compound Poisson-Gamma)
2. Time-based split (2021-2023 train, 2024 validate) mimics real-world temporal prediction
3. No k-fold cross-validation (deferred per user plan)
4. No hyperparameter tuning (fixed from v3 parameters)
5. Target encoding uses training data only (no leakage)
6. GLM assumes log-link is appropriate (standard for insurance pricing)
7. Blend weights (50/50, 70/30) are heuristic, not optimised

### Data Quality
1. Claims with incurred > 100x median are genuine large claims, not data errors
2. sum_insured x1000 errors identified and corrected for buildings/contents
3. All ID columns stripped of whitespace for consistent joins
4. construction_type normalised to uppercase with W/T combined as TIMBER
5. motor customer_id zero-padded to 8 characters for customer joins

## Future Improvements

### Modelling
* **K-fold cross-validation**: Add for more robust model validation (deferred per user plan)
* **Hyperparameter tuning**: Grid search or Bayesian optimisation for XGBoost and GLM parameters
* **Optimised blend weights**: ✅ Done — searched 0.20 to 0.80 in 0.05 steps, found per-line optimal (motor 0.60, buildings 0.40, contents 0.35)
* **Separate frequency-severity models**: Decompose into claim frequency (logistic/Poisson) and severity (Gamma) models for more interpretable pricing
* **Large claims modelling**: Separate treatment for motor large claims (e.g., cap at 99th percentile, model separately)
* **Categorical grouping**: Group age bands, years licensed bands to prevent overfitting and improve GLM interpretability
* **Interaction terms**: Add interaction features (e.g., age x vehicle_use, flood_zone x sum_insured) for GLM
* **Chi-squared testing**: Statistical significance testing for categorical features before inclusion in GLM

### Data & Features
* **Hypothesis testing**: Use GLM p-values to identify and remove non-significant rating factors
* **Feature engineering**: Create polynomial features, interaction terms, time-based features (month, quarter)
* **Better target encoding**: Use k-fold target encoding to reduce overfitting from in-sample encoding
* **Weather feature engineering**: Create lagged weather features, cumulative rainfall, extreme weather indicators

### Claims Development
* **Data-driven chain-ladder**: ✅ Done — derived per-peril factors from 2021 mature year (BI=1.82, TPP=1.08, subsidence=1.47)
* **Per-peril BF factors**: ✅ Done — six separate tail categories instead of three
* **IBNR reserve analysis**: More formal actuarial reserve analysis with uncertainty ranges

### Pipeline
* **Automated QA**: Add automated data quality checks and alerts for anomalies
* **Model monitoring**: Track prediction drift between training and test periods
* **Ensemble methods**: Add more model types (random forest, neural network) to the blend

## Files Inventory

### Notebooks (notebooks_v4/)
* `01_Data_Prep_and_Claims.ipynb` — Data cleaning, enrichment, claims development, seasonality/trend
* `02_XGBoost_Tweedie.ipynb` — Feature selection, XGBoost training, validation, submission
* `03_GLM_and_Blend.ipynb` — GLM training, blend, submissions
* `04_Decision_Log.ipynb` — This notebook (audit trail)

### Submissions (submissions_v4/)
* `xgboost_tweedie_v4.csv` — XGBoost only (mean=254.93)
* `glm_tweedie_v4.csv` — GLM only (mean=470.10)
* `blend_5050_v4.csv` — 50/50 blend (mean=357.15)
* `blend_7030_v4.csv` — 70/30 blend (mean=326.19)
* `blend_optimal_w0.45_v4.csv` — Overall best single weight (mean=337.52)
* **`blend_per_line_optimal_v4.csv` — Per-line optimal blend, recommended** (mean=339.10, D2=0.0844)

### Excel Checkpoints (Excel files_v4/)
* `claims_development_summary.xlsx` — 6 sheets (dev factors, claims rates, seasonality, trend, triangle, QA)
* `xgboost_v4_results.xlsx` — D2 scores + feature importance per line
* `glm_blend_v4_results.xlsx` — Model comparison + GLM coefficients per line
* `Features_v2.xlsx` — Original feature selection spreadsheet (3 sheets: Motor, Contents, Building)

### Processed Data (data/proc/)
* `proc_motor.parquet` — Motor training data with claims and enrichment
* `proc_buildings.parquet` — Buildings training data with claims and enrichment
* `proc_contents.parquet` — Contents training data with claims and enrichment
* `xgb_val_pred_{line}.parquet` — XGBoost validation predictions (for blend)